# Assignment 3 — Fine-Tuning I: Teaching a Model a Specific Taxonomy

Banks receive customer messages that need to be routed to the right internal team.
**BANKING77** (Casanueva et al., 2020) is a real dataset of 13,083 customer service
messages labeled with 77 fine-grained intents. Here we'll work with a 10-intent
subset, all related to cards and card payments — chosen because they're genuinely
hard to tell apart from wording alone.

**Before you start:**
- Make sure your JupyterHub server is running on a **GPU**, not CPU-only.
- Cells marked `# YOUR CODE HERE` are yours to write — these are the actual
  assignment, not just parameter tweaks. Everything else is scaffolding.
- Your written answers go in the **OLAT form**, not here — but keep this
  notebook's outputs visible, since you'll be reading numbers off them.


## Setup (provided — no changes needed here)

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes


In [ ]:
import time
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("No GPU detected — check your JupyterHub server settings before continuing.")


In [ ]:
LABELS = [
    "card_arrival", "card_not_working", "card_swallowed", "card_about_to_expire",
    "card_linking", "activate_my_card", "lost_or_stolen_card", "declined_card_payment",
    "card_payment_not_recognised", "compromised_card",
]

raw = load_dataset("PolyAI/banking77")
label_names = raw["train"].features["label"].names

def to_df(split):
    df = pd.DataFrame(split)
    df["label"] = df["label"].apply(lambda i: label_names[i])
    return df[df["label"].isin(LABELS)][["text", "label"]]

full_train = to_df(raw["train"])
full_test = to_df(raw["test"])

train_df = (
    full_train.groupby("label", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 30), random_state=SEED))
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
test_df = full_test.sample(n=50, random_state=SEED).reset_index(drop=True)

print(f"Train set: {len(train_df)} examples")
print(f"Test set:  {len(test_df)} examples")
train_df.head()


In [ ]:
# Just for your own reference before training -- are the 10 intents roughly
# balanced? Not something you need to report, but worth a glance before Task 5.
train_df["label"].value_counts()


## Load the model (provided)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
base_model.eval()
print("Model loaded.")


---
## Task 1 — The Prompting Baseline

Write a zero-shot classification prompt: give the model a customer message and
the list of 10 allowed intents, and ask it to pick one. Then measure accuracy
on the 50-example test set.


In [ ]:
def build_prompt(message):
    """
    YOUR CODE HERE.

    Build a prompt string that:
    - tells the model it needs to classify `message` into one of the 10 LABELS
    - lists the LABELS so the model knows what's allowed
    - instructs it to respond with only the category name (nothing else --
      you don't want to have to parse a full sentence back out)

    Return the prompt as a single string.
    """
    raise NotImplementedError("Write your zero-shot classification prompt here.")


In [ ]:
def classify(message, model_to_use):
    """
    YOUR CODE HERE.

    Use `build_prompt(message)` to build the prompt, wrap it as a chat message
    (role "user"), tokenize it with `tokenizer.apply_chat_template(...,
    add_generation_prompt=True)`, run `model_to_use.generate(...)` with a small
    `max_new_tokens` (you don't need many -- it's just a label), and decode
    only the NEW tokens (not the prompt echoed back) with
    `tokenizer.decode(..., skip_special_tokens=True)`.

    Return the decoded prediction string, stripped of whitespace.
    """
    raise NotImplementedError("Write the classify() function here.")


In [ ]:
def evaluate(model_to_use, df):
    """
    Provided -- this just loops classify() over a dataframe and computes accuracy.
    A prediction counts as correct if the gold label string appears anywhere in
    the model's output (loose matching -- the base model may not always answer
    with exactly the bare label).
    """
    rows = []
    for _, row in df.iterrows():
        prediction = classify(row["text"], model_to_use)
        rows.append({
            "text": row["text"],
            "gold": row["label"],
            "prediction": prediction,
            "correct": row["label"] in prediction,
        })
    results = pd.DataFrame(rows)
    accuracy = results["correct"].mean() * 100
    return accuracy, results


In [ ]:
baseline_accuracy, baseline_results = evaluate(base_model, test_df)
print(f"Baseline accuracy: {baseline_accuracy:.1f}%")


In [ ]:
# Pick 3 mistakes here to write about in your OLAT answer. What kind of
# confusion is happening -- are the predicted and gold labels semantically
# close (e.g. both about a malfunctioning card)?
mistakes = baseline_results[~baseline_results["correct"]]
mistakes.head(10)


---
## Task 2 — Prepare the Fine-Tuning Data

Format each training example as an instruction-following pair: the same prompt
style as Task 1, with the correct label as the target completion. Just the label
itself (e.g. `card_arrival`) — no JSON wrapper needed, since there's only one
output field.


In [ ]:
def format_example(row):
    """
    YOUR CODE HERE.

    Return a dict with two keys:
      - "prompt": the same prompt you'd get from build_prompt(row["text"])
      - "completion": the target completion the model should learn to produce
        -- just the label string (row["label"]), formatted so it tokenizes
        naturally after your prompt's "...Category:" ending (a leading space
        usually matters here -- think about why).

    Example return value:
        {"prompt": "...Category:", "completion": " card_arrival"}
    """
    raise NotImplementedError("Write format_example() here.")


In [ ]:
formatted_train = [format_example(row) for _, row in train_df.iterrows()]
train_dataset = Dataset.from_list(formatted_train)
formatted_train[0]  # sanity check -- does this look right?


---
## Task 3 — LoRA Fine-Tuning on a T4

Fill in the LoRA config below, then train for 3 epochs. The training loop
itself is provided -- your job is the config.


In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# YOUR CODE HERE: pick a rank (r) and target_modules.
# r=8 targeting q_proj/v_proj is a reasonable default for a model this size on
# this much data -- you can keep it, or justify something different in your
# OLAT answer (what would a higher rank or different target modules trade off?).
lora_config = LoraConfig(
    r=None,                 # TODO
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=None,    # TODO, e.g. ["q_proj", "v_proj"]
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Provided -- the training loop itself isn't the point of this task.
training_args = SFTConfig(
    output_dir="./banking77-lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    report_to=[],
)

trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_dataset)

start = time.time()
trainer.train()
elapsed_minutes = (time.time() - start) / 60
print(f"Training took {elapsed_minutes:.1f} minutes")


In [ ]:
# Report the final training loss (from the log above) and the training time
# (printed above) in your OLAT answer.


---
## Task 4 — Evaluate the Fine-Tuned Model

Reuse your own `evaluate()` function from Task 1, same test set, now on the
fine-tuned model.


In [ ]:
finetuned_accuracy, finetuned_results = evaluate(model, test_df)

print(f"Base model accuracy:      {baseline_accuracy:.1f}%")
print(f"Fine-tuned model accuracy: {finetuned_accuracy:.1f}%")


In [ ]:
# Look at the same 3 mistakes you picked in Task 1. Does the fine-tuned model
# get them right now? Report what changed in your OLAT answer.
comparison = baseline_results.merge(
    finetuned_results, on="text", suffixes=("_base", "_finetuned")
)
comparison[comparison["text"].isin(mistakes["text"])][
    ["text", "gold_base", "prediction_base", "prediction_finetuned", "correct_finetuned"]
]


---
## Task 5 — Overfitting

Re-run fine-tuning from scratch, this time for 15 epochs instead of 3. Compare
training accuracy (on the 300 examples the model trained on) against test
accuracy (on the same 50 held-out examples as always).


In [ ]:
# Start from the original base model again -- not from the Task 3 checkpoint --
# so this is a clean, independent comparison. Reuses your lora_config from Task 3.
fresh_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
overfit_model = get_peft_model(fresh_base_model, lora_config)

overfit_args = SFTConfig(
    output_dir="./banking77-lora-overfit",
    num_train_epochs=15,   # the only change from Task 3
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    report_to=[],
)

overfit_trainer = SFTTrainer(model=overfit_model, args=overfit_args, train_dataset=train_dataset)
overfit_trainer.train()


In [ ]:
train_accuracy_15ep, _ = evaluate(overfit_model, train_df)
test_accuracy_15ep, _ = evaluate(overfit_model, test_df)

train_accuracy_3ep, _ = evaluate(model, train_df)
test_accuracy_3ep = finetuned_accuracy

print("               Training acc.   Test acc.")
print(f"3 epochs:      {train_accuracy_3ep:>6.1f}%       {test_accuracy_3ep:>6.1f}%")
print(f"15 epochs:     {train_accuracy_15ep:>6.1f}%       {test_accuracy_15ep:>6.1f}%")


In [ ]:
# Fill these four numbers into the Task 5 table on OLAT, and answer the
# reflection question about the train/test gap.


---
## Task 6 — When Is Fine-Tuning Worth It?

No code here. Head back to the OLAT form and answer using the real numbers
you've gathered above.
